### Nearest Neighbor Classifier: The "Lazy" Algorithm

#### 1. Instance-Based Learning (Ye Model Kyu Nahi Banata?)
Ab tak humne jo bhi padha (Logistic, Ridge), wo **Model-Based Learning** the. Unme algorithm training data ko dekhta tha, math lagata tha, aur ek equation (jaise $y = w^Tx + b$) ke weights ($w$) learn karta tha. Ek baar weights mil gaye, toh wo training data ko RAM se delete (bhool) jata tha.

Lekin Nearest Neighbor ek **Instance-Based Learning** (ya Non-Generalizing / Lazy Learning) hai. 
* **Lazy kyu?** Kyunki ye training phase mein kuch bhi "seekhta" ya "math" nahi karta. Ye chup-chap saare training data (instances) ko RAM mein store/ratt leta hai. 
* Iska `fit(x_train, y_train)` time complexity lagbhag $\mathcal{O}(1)$ hota hai (kyunki sirf data save kar raha hai).
* Lekin iska `predict()` bohot slow (computationally expensive) hota hai, kyunki jab bhi koi naya data aayega, isko poore dataset ke sath distance calculate karna padega.

#### 2. The Core Math: Distance Metrics
Classification karne ke liye isko naye point (test data) aur purane points (training data) ke beech ki doori (distance) nikalni hoti hai. Scikit-learn by default **Minkowski Distance** use karta hai:
$$D(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{\frac{1}{p}}$$

Ye formula ek "Master Formula" hai. Isme $p$ ki value change karne se distance ka type badal jata hai:
* **Agar $p=1$ (Manhattan Distance / L1 Norm):** Ye city-block ki tarah grid par chalta hai. Equation: $D = \sum |x_i - y_i|$
* **Agar $p=2$ (Euclidean Distance / L2 Norm):** Ye aam zindagi ka "straight line" distance hai (Pythagoras theorem). Equation: $D = \sqrt{\sum (x_i - y_i)^2}$

#### 3. Classification: The Majority Vote
Jab hum naya data point daalte hain, algorithm 3 steps karta hai:
1. Naye point ka har ek training point se distance nikalta hai.
2. Unhe ascending order (sabse paas se sabse door) mein sort karta hai.
3. Top $K$ (nearest) doston ko uthata hai.

**Math of Probability:**
Agar humare $K$ neighbors mein $C$ alag-alag classes hain, toh probability of class $j$ ye hoti hai:
$$P(y = j | X) = \frac{1}{K} \sum_{i=1}^{K} I(y^{(i)} = j)$$
*(Jahan $I$ ek indicator function hai jo $1$ hota hai agar neighbor ki class $j$ hai, warna $0$)*. Jiska score/count sabse high hoga, naya point usi class ka ghoshit ho jayega.

#### 4. KNeighborsClassifier vs RadiusNeighborsClassifier

Scikit-learn mein iske do alag-alag algorithms (implementations) hain. Inka difference samajhna advanced ML ka part hai:

**A. KNeighborsClassifier (The Fixed Count)**
* **Logic:** Isme tum $K$ ki value (jaise 5) fix kar dete ho. 
* **Behavior:** Naya point apne aas-paas tab tak radius badhata jayega jab tak usko theek 5 log nahi mil jate. Agar data dense (ghana) hai, toh 5 log bohot paas mil jayenge. Agar data sparse (khali) hai, toh 5 log dhundhne ke liye ye bohot door tak apna circle bada karega.
* **Flaw:** Outliers wale area mein ye bohot door ke unrelated points ko bhi apna "neighbor" maan leta hai.

**B. RadiusNeighborsClassifier (The Fixed Distance)**
* **Logic:** Isme hum log/count ($K$) fix nahi karte, balki **Radius ($R$) fix karte hain** (jaise 2 units distance).
* **Behavior:** Naya point sirf 2 units ka ek circle banata hai. Agar us circle mein 50 log aaye, toh 50 ka vote lega. Agar 2 log aaye, toh 2 ka vote lega.
* **Advantage:** Ye ajeeb (non-uniform) density wale data par $KNN$ se better perform karta hai.
* **Flaw:** Agar circle mein *zero* points aaye, toh model crash ho sakta hai (iske liye outlier label set karna padta hai).

#### 5. Advance Pre-requisite: Feature Scaling is COMPULSORY
Kyunki KNN puri tarah se "Distance Math" (Subtraction aur Squares) par based hai, agar tumhara ek feature "Salary" ($100,000$) mein hai aur dusra feature "Age" ($25$) mein hai, toh Salary ka distance Age ke distance ko completely daba dega (overpower kar dega).
Isliye KNN lagane se pehle `MinMaxScaler` ya `StandardScaler` lagana 100% zaroori hai, warna model fail ho jayega.

---

### KNeighbors vs. RadiusNeighbors: Asli Farak Kya Hai?

Dono algorithms ka basic funda ek hi hai: **"Jo point naye data ke paas hoga, wahi class output hogi."** Lekin "Paas koun hai?" isko define karne ke inke tarike ekdum alag hain. 

Aao isko ekdum deeply samajhte hain ki industry mein kisko kab choose kiya jata hai.

#### 1. KNeighborsClassifier (The "Fixed Count" Approach)
* **Logic:** Ye algorithm ziddi hota hai. Tumne agar $K=5$ set kar diya, toh isko exactly 5 padosi (neighbors) chahiye hi chahiye vote karne ke liye. 
* **Behavior:** Naya data point center par khada hota hai aur apna radar (distance) tab tak badhata rehta hai jab tak use exactly 5 training points nahi mil jate.
* **Flaw (Problem):** Socho tumhara data ekdum alag-alag faila hua hai (sparse). Naya point aisi jagah land karta hai jahan aas-paas koi nahi hai. Kyunki isko 5 log chahiye hi chahiye, ye apna circle itna bada kar lega ki ye bohot door baithe galat class walo ko bhi apna "neighbor" maan lega. Isse prediction galat ho jati hai. 
* **Use Case:** Jab tumhara data "Uniformly Sampled" ho (matlab har jagah points barabar density mein faile hon), tab ye best aur sabse commonly use hota hai.

#### 2. RadiusNeighborsClassifier (The "Fixed Area" Approach)
* **Logic:** Ye algorithm count ($K$) fix nahi karta, ye **Radius ($r$)** fix karta hai. Tum isko bolte ho: "Bhai, sirf 2.0 unit ke distance (circle) mein dekhna, uske bahar nahi."
* **Behavior:** Naya point sirf utne fixed radius ka ek circle draw karta hai. Ab us circle ke andar agar 50 points aa gaye, toh wo 50 ka vote lega. Agar us sparse (khali) area mein sirf 2 points aaye, toh wo sirf un 2 points ka vote lega. 
* **Advantage (Non-Uniform Data):** Lecture mein jo likha hai *"used in cases where the data is not uniformly sampled"*, uska matlab yahi hai. Agar data kisi area mein bohot ghana (dense) hai aur kisi area mein ekdum khali (sparse), toh Radius Classifier smart behave karta hai. Khali area mein ye zabardasti door ke points nahi uthata, balki kam neighbors se hi decision le leta hai.

#### 3. Real-Life Analogy se Samajhte Hain
Maan lo tumhe PUBG mein push karna hai ya nahi, iski advice leni hai.
* **KNeighbors (K=4):** Tum bolte ho "Main exactly 4 teammates ki baat sununga." Ab 1 teammate tumhare paas hai, par baaki 3 teammates map ke dusre kone mein hain. Tum zabardasti unki baat sunkar push kar doge aur mar jaoge (Wrong classification due to outliers).
* **RadiusNeighbors (r=50m):** Tum bolte ho "Main sirf unki sununga jo mere 50 meter ke radius mein hain." Agar wahan sirf 1 teammate hai, toh tum sirf usi 1 teammate ki advice par decision loge, unki nahi jo door hain. Ye zyada logical hai!

#### 4. The Mathematical Catch (Outlier Problem)
RadiusNeighborsClassifier theoretically zyada smart lagta hai, par isme ek bada math ka issue aata hai jise handle karna padta hai: **The Empty Circle.**

Mathematically, equation aisi banti hai: 
$$P(y=j | X) = \frac{1}{N} \sum_{i \in \text{Radius}} I(y^{(i)} = j)$$
*(Jahan $N$ un points ka count hai jo Radius ke andar aaye hain).* Lekin kya hoga agar tumhara naya test point ek aisi ajeeb jagah aa jaye jahan radius $r$ ke andar **ZERO** points aayen? Tab $N=0$ ho jayega. Math mein $\frac{1}{0}$ Error/Infinity ban jata hai aur tumhara Python code wahi crash ho jayega (ValueError). Isliye Scikit-learn mein is classifier ke andar ek special parameter hota hai `outlier_label`, jahan tum pehle hi bata dete ho ki "Agar koi points na mile, toh isko default 'Unknown' ya specific class mark kar dena."

---

In [10]:
import numpy as np
from pprint import pprint

np.random.seed(42)

from sklearn.datasets import fetch_openml

from sklearn.preprocessing import StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression , LogisticRegressionCV

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score, 
    make_scorer,
)

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# global settings
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)
mpl.rc('figure',figsize=(8,6))



from sklearn.datasets import fetch_openml
x_pd , y_pd = fetch_openml('mnist_784' , version=1 , return_X_y=True)
x = x_pd.to_numpy()
y = y_pd.to_numpy()

scaler = MinMaxScaler()
X = scaler.fit_transform(x)

target_names = np.unique(y)

x_train, x_test, y_train, y_test = X[ :60000], X[60000 : ] , y[ : 60000], y[ 60000 : ]

y_train_0 = np.zeros(len(y_train))
y_test_0 = np.zeros(len(y_test))

index_0 = np.where(y_train=='0')
y_train_0[index_0] = 1

index_0 = np.where(y_test=='0')
y_test_0[index_0] = 1

### KNN Weights: Uniform vs. Distance (The Math of Voting)

#### 1. Problem Kya Hai? (The Intuition)
Pichle notes mein humne padha tha ki KNN "Majority Vote" leta hai. Agar $K=5$ hai, toh jo class sabse zyada baar aayegi, model wahi predict karega. 

Par isme ek bahut badi practical problem hai. Maan lo tumhara ek naya point (Test Point) aaya. Uske aas-paas 5 log hain:
* 2 dost (Class 1) ekdum **chipke hue** khade hain (Distance bohot kam hai).
* 3 dushman (Class 0) door boundary par **door-door** khade hain.

Agar normal voting hogi, toh Class 0 jeet jayegi (3 votes > 2 votes). Lekin logically socho, jo 2 points ekdum tumhare paas hain, unki baat ka asar (influence) zyada hona chahiye na? Yahi par Scikit-Learn ka `weights` parameter kaam aata hai.

#### 2. weights='uniform' (The Default Democratic Voting)
* **Kaisa behave karta hai:** Ye default setting hai. Isme "One Man, One Vote" ka rule lagta hai. 
* **The Math:** Har neighbor ka weight mathematically $1$ set hota hai. Chahe wo padosi ekdum tumhare upar betha ho (Distance = 0.1) ya fir circle ke ekdum aakhiri kone par ho (Distance = 10.0), dono ka vote barabar count hoga.
* **Problem:** Agar data sparse (khali) hai aur tumne $K$ bada le liya, toh bohot door baithe hue ajeeb points (outliers) aakar tumhara result kharab kar denge.

#### 3. weights='distance' (The Advanced Smart Voting)
* **Kaisa behave karta hai:** Ye "Jo jitna paas, uski utni chalegi" wala rule hai. Isme vote ki value point ki doori par depend karti hai.
* **The Math:** Is setting mein Scikit-learn har vote ko distance ke inverse (ulje) se multiply kar deta hai. Formula:
$$W_i = \frac{1}{d_i}$$
*(Jahan $W_i$ neighbor $i$ ka weight/vote power hai, aur $d_i$ uski distance hai)*

Chalo iska ek numerical breakdown karte hain taaki dimaag mein math set ho jaye:
* **Neighbor A (Class 1) paas hai:** Distance $d = 0.5$. Iska weight kitna hua? $\frac{1}{0.5} = 2.0$ votes!
* **Neighbor B (Class 0) door hai:** Distance $d = 5.0$. Iska weight kitna hua? $\frac{1}{5.0} = 0.2$ votes!

Ab agar hum score total karenge, toh paas wale Neighbor A akele 10 Neighbor B ke barabar power rakhta hai. Isse distance wali exactness model mein wapas aa jati hai.

#### 4. Industry Standard Use Case
Real world ML problems mein, agar tumhe lagta hai ki tumhara dataset thoda messy hai ya classes ek dusre mein mix ho rahi hain (overlapping boundaries), toh `weights='distance'` hamesha `uniform` se better accuracy deta hai. Ye KNN ke sabse bade weakness (door ke points ka interference) ko fix kar deta hai.

---

### The 'algorithm' Parameter in KNN: Kaise Chunein?

KNN ka sabse bada disadvantage hai iski speed. Naya test point aate hi usko pata lagana hota hai ki sabse paas ke $K$ points kaunse hain. Ye pata lagane ke 3 alag-alag mathematical tarike (algorithms) hote hain: `'brute'`, `'kd_tree'`, aur `'ball_tree'`.

#### 1. Brute Force ('brute')
* **Logic:** Ye ekdum basic "mazdoori" wala tarika hai. Naya test point aayega, aur ye algorithm dataset ke **har ek single point** ke sath uska distance calculate karega. Fir sabko sort karke top $K$ nikalega.
* **The Math (Time Complexity):** Agar tumhare paas $N$ samples hain aur $D$ features/dimensions hain, toh iska search time $\mathcal{O}(N \times D)$ hota hai.
* **Kab use karna hai:** Jab dataset bohot chota ho (jaise $N < 1000$). Chote data mein tree banane ka time waste karne se acha hai direct distance nikal lo.

#### 2. KD Tree ('kd_tree' - K-Dimensional Tree)
* **Logic:** Har baar sabse distance nikalna bewakoofi hai. KD Tree data ko "Boxes" (rectangles) mein baant deta hai. Jaise ek dictionary mein 'M' se shuru hone wala word dhundhna ho, toh hum A-L wale pages skip kar dete hain. Waise hi KD Tree binary search jaisa math lagata hai, aur aadhe space ko bina distance calculate kiye reject kar deta hai.

* **The Math:** Iska search time bohot fast ho jata hai: $\mathcal{O}(D \log(N))$. 
* **The Problem (Curse of Dimensionality):** KD Tree tabhi kaam karta hai jab dimensions (columns/features) kam hon. Agar tumhare columns $20$ se zyada ho gaye, toh math itna complex ho jata hai ki ye Brute Force se bhi zyada slow ho jata hai!
* **Kab use karna hai:** Jab data bada ho, par features (columns) kam hon (jaise 2D ya 3D data).

#### 3. Ball Tree ('ball_tree')
* **Logic:** KD Tree ki high-dimension wali problem ko solve karne ke liye Ball Tree banaya gaya. Ye data ko square boxes mein baantne ki jagah, "Spheres" (Balls) ke andar baant-ta hai (Ek bade gole ke andar chote-chote gole). 
* **The Math:** Jab naya point aata hai, toh ye bas gole ke center ka distance aur radius check karta hai. Agar point radius ke bahar hai, toh us gole ke andar ke saare points ko ek jhatke mein reject kar deta hai. Iska search time bhi $\mathcal{O}(D \log(N))$ hai, par high dimensions mein math break nahi hoti.
* **Kab use karna hai:** Jab dataset bada ho AUR features/columns bhi bohot zyada hon.

#### 4. Auto ('auto') - The Ultimate Default
* **Logic:** Scikit-Learn ke developers ko pata tha ki data scientist baar-baar ye math calculate karke confuse hoga. Isliye default setting `'auto'` hoti hai.
* Ye kya karta hai? `.fit()` call hote hi algorithm khud data ka size, dimensions, aur sparsity check karta hai aur background mein automatically decide kar leta hai ki `'brute'`, `'kd_tree'`, ya `'ball_tree'` mein se kya best rahega.

### Summary / Cheat Sheet

| Parameter Value | Dataset Size | Number of Features (D) | Best Use Case |
| :--- | :--- | :--- | :--- |
| **'brute'** | Small | Any | Toy datasets, small arrays. |
| **'kd_tree'** | Large | Low ($< 20$) | Spatial data, simple tabular data. |
| **'ball_tree'** | Large | High ($> 20$) | Complex tabular data, images. |
| **'auto'** | Any | Any | **Industry standard.** Hamesha yahi use karo, model khud smart choice lega! |


---

### Advanced Parameters for Tree Algorithms in KNN

Jab tum `algorithm='kd_tree'` ya `ball_tree'` set karte ho, toh Scikit-Learn 3 aur important parameters par dhyan deta hai: `metric`, `p`, aur `leaf_size`.

#### 1. The 'metric' Parameter (Doori Napne Ka Tarika)
KNN puri tarah se "Distance" par zinda hai. Par distance nikalne ka sirf ek formula nahi hota. Scikit-learn tumhe options deta hai ki tumhara data kaisa hai us hisaab se math chuno:
* **'euclidean':** Ye normal straight-line distance hai (Pythagoras theorem).
* **'manhattan':** Ye grid jaisa distance hai (jaise city ke blocks cross karna).
* **'chebyshev':** Ye maximum absolute difference dekhta hai.
* **'minkowski' (Default):** Ye in sabka ek "Master Formula" (Baap) hai. 

#### 2. The 'p' Parameter (Minkowski's Remote Control)
Kyunki default metric `minkowski` hai, hum isko chalane ke liye ek extra variable dete hain jisko `p` (Power parameter) bolte hain. 

Minkowski distance ka main formula ye hota hai:
$$D(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{\frac{1}{p}}$$

Ye formula ek remote control ki tarah kaam karta hai. Tum `p` ki value change karke formula ko badal sakte ho:
* **Agar `p=1`:** Formula mein power $1$ ho jayegi, aur ye automatically **Manhattan Distance** ban jayega. 
* **Agar `p=2` (Default):** Formula mein power $2$ (square aur square root) ho jayegi, aur ye exactly **Euclidean Distance** ban jayega.
* *Note:* Ye parameter strictly tabhi kaam karta hai jab tumhara metric 'minkowski' set ho.

#### 3. 'leaf_size' (The Speed vs. Memory Trade-off)
Ye parameter Tree-algorithms (KD aur Ball tree) ka sabse technical hissa hai. 
Jab algorithm data ko baant kar Tree banata hai (bade boxes ke andar chote boxes), toh ek point aata hai jahan further divide karna bewaqoofi ho jati hai. 

**Intuition (Real-life Example):**
Maan lo tumhe ek dictionary mein koi word dhundhna hai. Tum A se M tak ke section banate ho, fir M ke andar alag sections. Lekin jab tumhare paas bas ek page (jaise 30 words) bachta hai, toh tum uske bhi half-sections nahi banate, tum us page ko line-by-line (brute-force) padh lete ho.

`leaf_size` (Default = 30) yahi limit set karta hai. 
* **Logic:** Ye algorithm ko bolta hai, "Jab branch/box ke andar strictly 30 ya usse kam points bachein, toh Tree banana band kar do. Un 30 points ke beech brute-force math laga kar answer nikal lena."

**Iska Model par kya farq padta hai?**
* **Chota `leaf_size` (e.g., 5):** Tree bohot bada aur deep banega. Isey RAM (Memory) bohot zyada chahiye hogi, aur model ka `.fit()` hone mein time lagega. Par naya data aane par search thoda fast ho sakta hai.
* **Bada `leaf_size` (e.g., 100):** Tree chota banega. Memory kam khaega aur turant `.fit()` (construct) ho jayega. Par search ke time jab ye leaf par pahuchega, toh isko 100 points ke sath mazdoori (brute-force distance calculate) karni padegi, jisse prediction thodi slow ho jayegi.
* *Industry Standard:* Zyadatar default `30` best balance deta hai CPU aur RAM ke beech mein.

---

In [12]:
from sklearn.neighbors import KNeighborsClassifier
kneighbors_classifier = KNeighborsClassifier(n_neighbors=3)
kneighbors_classifier.fit(x_train , y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None
